# Titanic Survival Prediction

A complete machine learning classification project using the Titanic passenger dataset.

**Goal:** Predict whether a passenger survived based on demographic, travel, and ticket-related features.

This notebook covers:
- Data loading and inspection
- Data cleaning
- Exploratory data analysis (EDA)
- Feature engineering and selection
- Train/test splitting
- Multiple classification models
- Model evaluation
- Confusion matrix, ROC curve, and feature importance
- Final observations and limitations


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, RocCurveDisplay
)

RANDOM_STATE = 42


In [2]:
# Load dataset
df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Basic inspection
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
display(df["Survived"].value_counts().to_frame("count"))


Data types:


,dtype
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64



Missing values:


,missing_values
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0



Duplicate rows: 0

Target distribution:


,count
Survived,
0,549
1,342


## Data Cleaning and Feature Engineering

The original notebook filled `Age`, converted `Sex` to numeric values, and removed `Cabin`.  
For a more robust GitHub-ready workflow, preprocessing is handled inside a scikit-learn pipeline so that transformations are learned from the training data only.

Additional features:
- `FamilySize = SibSp + Parch + 1`
- `IsAlone = 1` when the passenger travelled alone
- `Title` extracted from the passenger name

`PassengerId`, `Name`, `Ticket`, and `Cabin` are not used directly as model features. `PassengerId` is an identifier rather than a meaningful predictive variable, while raw `Name`, `Ticket`, and `Cabin` are replaced or excluded to keep the project focused.


In [4]:
# Feature engineering
data = df.copy()

data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

data["Title"] = (
    data["Name"]
    .str.extract(r",\s*([^.]*)\.", expand=False)
    .str.strip()
)

# Group uncommon titles to reduce sparse categories
common_titles = ["Mr", "Miss", "Mrs", "Master"]
data["Title"] = data["Title"].where(data["Title"].isin(common_titles), "Other")

display(data[["Name", "Title", "FamilySize", "IsAlone"]].head())


,Name,Title,FamilySize,IsAlone
0,"Braund, Mr. Owen Harris",Mr,2,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0
2,"Heikkinen, Miss. Laina",Miss,1,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0
4,"Allen, Mr. William Henry",Mr,1,1


## Exploratory Data Analysis

In [5]:
# Survival counts
survival_counts = data["Survived"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(["Did not survive", "Survived"], survival_counts.values)
plt.title("Titanic Survival Distribution")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()


In [6]:
# Survival rate by sex
sex_survival = data.groupby("Sex")["Survived"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(sex_survival.index, sex_survival.values)
plt.title("Survival Rate by Sex")
plt.ylabel("Survival Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


In [7]:
# Survival rate by passenger class
class_survival = data.groupby("Pclass")["Survived"].mean()

plt.figure(figsize=(7, 4))
plt.bar(class_survival.index.astype(str), class_survival.values)
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


In [8]:
# Age distribution
plt.figure(figsize=(8, 4))
plt.hist(data["Age"].dropna(), bins=25)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()


In [9]:
# Correlation among selected numeric variables
corr_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
corr = data[corr_cols].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## Feature Selection and Preprocessing

The target is `Survived`.

Selected features:
- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`
- `FamilySize`
- `IsAlone`
- `Title`

Numeric missing values are median-imputed and standardized. Categorical missing values are filled with the most frequent category and one-hot encoded. The preprocessing pipeline is fitted only on the training data to avoid data leakage.


In [10]:
features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "FamilySize", "IsAlone", "Title"
]

target = "Survived"

X = data[features]
y = data[target]

numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone"
]
categorical_features = ["Sex", "Embarked", "Title"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 712
Testing rows: 179


## Model Training

In [11]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE
    )
}

results = []
fitted_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(4))


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.8436,0.8254,0.7536,0.7879,0.8792
2,Gradient Boosting,0.8156,0.8103,0.6812,0.7402,0.8501
1,Random Forest,0.8212,0.8246,0.6812,0.7460,0.8431


In [12]:
# Metric comparison
plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]

ax = plot_df.plot(kind="bar", figsize=(11, 5))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [13]:
# Confusion matrix and classification report for Logistic Regression
logistic_model = fitted_models["Logistic Regression"]
logistic_pred = logistic_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, logistic_pred)
plt.title("Logistic Regression - Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_test, logistic_pred))


              precision    recall  f1-score   support

           0       0.85      0.90      0.88       110
           1       0.83      0.75      0.79        69

    accuracy                           0.84       179
   macro avg       0.84      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179



In [14]:
# ROC curves
plt.figure(figsize=(8, 6))

for name, pipeline in fitted_models.items():
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, probabilities, name=name, ax=plt.gca())

plt.title("ROC Curves")
plt.tight_layout()
plt.show()


In [15]:
# Feature importance from Random Forest
rf_pipeline = fitted_models["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]

feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .head(15)
)

plt.figure(figsize=(9, 6))
plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

display(importance_df)


,Feature,Importance
4,numeric__Fare,0.185213
1,numeric__Age,0.147101
14,categorical__Title_Mr,0.131340
8,categorical__Sex_male,0.127927
7,categorical__Sex_female,0.097444
0,numeric__Pclass,0.090717
5,numeric__FamilySize,0.055174
15,categorical__Title_Mrs,0.032829
2,numeric__SibSp,0.028451
13,categorical__Title_Miss,0.027165


## Conclusion

This project demonstrates an end-to-end binary classification workflow for Titanic survival prediction.

Key practices used:
- Missing-value handling through a reproducible preprocessing pipeline
- Feature engineering with family size, travelling-alone status, and title
- Stratified train/test split
- Comparison of three classification algorithms
- Multiple evaluation metrics rather than accuracy alone
- Confusion matrix and ROC curve analysis
- Feature importance inspection

The reported metrics are based on this notebook's fixed 80/20 test split and should not be treated as a guarantee of performance on new datasets. The Titanic dataset is historical and relatively small, so the results are primarily useful for demonstrating the machine-learning workflow.
